In [ ]:
import hashlib,importlib.metadata,json,os,signal,subprocess,sys,time,urllib.request
from pathlib import Path
ROOT=Path('/kaggle/working');WORK=Path('/tmp')/('mgbfs-collective-'+str(os.getpid()));WORK.mkdir()
BASE='https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261016-v22/'
m=json.loads(urllib.request.urlopen(BASE+'manifest.json',timeout=60).read());assert m['sha256']=='b691739ae60e917a7a4a677453edc666b9aaa38a34b29931eb0db37deda2a267' and m['size']<2_000_000
wheel=WORK/m['wheel'];urllib.request.urlretrieve(BASE+m['wheel'],wheel);assert hashlib.sha256(wheel.read_bytes()).hexdigest()==m['sha256']
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)],check=True)
subprocess.run([sys.executable,'-m','pip','install','cayleypy==0.2.0'],check=True)
assert importlib.metadata.version('multigpubfs')=='0.2.0.dev22'
raw=subprocess.check_output(['nvidia-smi','--query-gpu=name,compute_cap,driver_version','--format=csv,noheader'],text=True)
hardware=[{'name':s.split(',')[0].strip(),'cc':[int(v) for v in s.split(',')[1].strip().split('.')],'driver':s.split(',')[2].strip()} for s in raw.splitlines() if s.strip()]
assert len(hardware)==2 and all(x['cc']==[7,5] for x in hardware),hardware
os.environ['MGBFS_NATIVE_CACHE']=str(WORK/'cache')
import multigpubfs
preloaded=multigpubfs.prefetch_native(sm=[75]);assert preloaded['status']=='PREFETCHED'
os.environ['MGBFS_NATIVE_OFFLINE']='1'
receipt={'schema':1,'status':'RUNNING','package':'0.2.0.dev22','package_source_commit':m['source_commit'],'wheel_sha256':m['sha256'],'hardware':hardware,'checks':[],
 'scope':'Physical 2xT4 collective installed-package diagnostics; no physical eight-GPU acceptance is claimed. No CUDA/Rust changes.'}
started=time.monotonic()
def record(name,**data):
 receipt['checks'].append(dict(name=name,status='PASS',**data));(ROOT/'collective-gpu-acceptance.json').write_text(json.dumps(receipt,indent=2));print('COLLECTIVE_GPU_CHECK',json.dumps(receipt['checks'][-1]),flush=True)
try:
 before=time.monotonic();out=WORK/'doctor'
 p=subprocess.Popen([sys.executable,'-m','multigpubfs','doctor','--output',str(out),'--stage-timeout','60'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=True)
 try:log,_=p.communicate(timeout=360)
 finally:
  if p.poll() is None:
   os.killpg(p.pid,signal.SIGTERM)
   try:p.wait(timeout=2)
   except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait(timeout=2)
 (ROOT/'collective-doctor-console.log').write_text(log);print(log[-14000:],flush=True)
 r=json.loads((out/'report.json').read_text());assert p.returncode==0 and r['status']=='PASS',r.get('failed_components')
 assert r['devices']==[0,1] and r['working_configuration']['status']=='READY' and r['working_configuration']['selected']['devices']==[0,1]
 gpu=[s for s in r['steps'] if s.get('diagnostic_case',{}).get('devices')]
 assert gpu and all(s['diagnostic_case']['devices']==[0,1] and 'device' not in s['diagnostic_case'] for s in gpu)
 group=[]
 for s in gpu:
  if s['name'].startswith('all-gpus-'):
   data=s['result']['data'];assert data['devices']==[0,1] and len(data['ranks'])==2 and all(x['status']=='PASS' for x in data['ranks'])
   group.append({'name':s['name'],'seconds':s['seconds'],'ranks':[{'rank':x['rank'],'device':x['device'],'status':x['status']} for x in data['ranks']]})
 record('ordinary-doctor-all-selected-gpus-no-single-gpu-sweeps',wall_seconds=time.monotonic()-before,steps=[{'name':s['name'],'status':s['status'],'seconds':s.get('seconds')} for s in r['steps']],joint_primitives=group,working_devices=[0,1],no_single_gpu_fallback=True)
 from multigpubfs import deep_diagnostics as deep
 from multigpubfs.diagnostics import Runner
 catalog=deep.catalog_configurations();runner=Runner(WORK/'catalog',60);runner.deep_capture=True;before=time.monotonic()
 result=deep.run_case(runner,{'name':'catalog-all-gpus','config':{'kind':'catalog-batch','devices':[0,1],'catalog':catalog},'seconds':60},time.monotonic()+180)
 print('CATALOG_RESULT',json.dumps({k:result[k] for k in ('name','status','seconds','error') if k in result}),flush=True)
 if result['status']!='PASS':print(json.dumps(result,default=str)[-25000:],flush=True)
 assert result['status']=='PASS';data=result['result']['data'];assert data['devices']==[0,1] and len(data['ranks'])==2
 ranks=[]
 for rank in data['ranks']:
  value=rank['result']['data'];assert value['status']=='PASS' and len(value['cases'])==len(catalog) and value['passed']>0 and value['failed']==0
  ranks.append({'rank':rank['rank'],'device':rank['device'],'pid':rank['pid'],'passed':value['passed'],'skipped':value['skipped'],'failed':value['failed'],'cases':[{'family':x['family'],'constructor':x['constructor'],'status':x['status']} for x in value['cases']]})
 record('entire-catalog-in-two-concurrent-reused-contexts',wall_seconds=time.monotonic()-before,catalog_cases=len(catalog),ranks=ranks)
 receipt.update(status='PASS',wall_seconds=time.monotonic()-started);(ROOT/'collective-gpu-acceptance.json').write_text(json.dumps(receipt,indent=2));print('COLLECTIVE_GPU_ACCEPTANCE_PASS',json.dumps(receipt),flush=True)
except BaseException as error:
 receipt.update(status='FAIL',error=str(error),exception_type=type(error).__name__);(ROOT/'collective-gpu-acceptance.json').write_text(json.dumps(receipt,indent=2));raise
